In [ ]:
from gpiozero import Buzzer, Button
import io
import time
import picamera
import smtplib
from flask import Flask, Response
from threading import Thread
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# Initialize Flask application
app = Flask(_name_)

# Initialize IR sensor, buzzer, and camera
ir_sensor = Button(5)  # Replace 4 with the correct GPIO pin number
buzzer = Buzzer(12)   # Replace 12 with the correct GPIO pin number
camera = picamera.PiCamera()
camera.rotation = 180
camera.resolution = (640, 480)
camera.framerate = 24

# Generator function to continuously generate video frames for livestreaming
def generate_frames():
    stream = io.BytesIO()
    for _ in camera.capture_continuous(stream, 'jpeg', use_video_port=True):
        stream.seek(0)
        yield (b'--frame\r\nContent-Type: image/jpeg\r\n\r\n' + stream.read() + b'\r\n')
        stream.seek(0)
        stream.truncate()

# Route to serve the video feed
@app.route('/video_feed')
def video_feed():
    return Response(generate_frames(), mimetype='multipart/x-mixed-replace; boundary=frame')

# Function to send an email with the recorded video
def send_email(subject, body, to_email, filename):
    from_email = 'katchimydeen001@gmail.com'  # Your email address
    from_password = 'decg vwaa yflu bamo'   # Your email password
    smtp_server = 'smtp.gmail.com'         # SMTP server address
    smtp_port = 587                          # Port for TLS

    msg = MIMEMultipart()
    msg['From'] = from_email
    msg['To'] = to_email
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    with open(filename, 'rb') as attachment:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header(
            'Content-Disposition',
            f'attachment; filename={filename}',
        )
        msg.attach(part)

    try:
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()  # Upgrade to a secure TLS connection
            server.login(from_email, from_password)
            server.send_message(msg)
            print(f"Email sent to {to_email} with attachment {filename}")
    except Exception as e:
        print(f"Failed to send email: {e}")

# Function to handle IR sensor detection
def detect_ir():
    while True:
        if ir_sensor.is_pressed:
            print("IR sensor triggered!")
            buzzer.on()  # Turn on the buzzer
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"/home/project/Documents/Motion Detector/Intruder_{timestamp}.h264"
            camera.start_recording(filename)
            time.sleep(10)  # Record video for 10 seconds
            camera.stop_recording()
            buzzer.off()  # Turn off the buzzer
            print(f"Video has been recorded: {filename}")

            # Send the recorded video via email
            email_subject = "IR Sensor Triggered: Video Recording"
            email_body = "An IR sensor was triggered. Please find the recorded video attached."
            recipient_email = "darshanarajasekar@gmail.com"  # Replace with the recipient's email address
            send_email(email_subject, email_body, recipient_email, filename)
        else:
            # Print no IR sensor trigger message
            print("No IR sensor trigger detected.")
            time.sleep(1)  # Check the sensor every second

# Main block to run the Flask server and handle IR sensor detection
if _name_ == '_main_':
    # Start the IR sensor detection thread
    ir_thread = Thread(target=detect_ir)
    ir_thread.start()
    
    # Start the Flask application
    app.run(host='0.0.0.0', port=5000, threaded=True)
